In [5]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.integrate import quad

# 1. Load C-MAPSS Data (Assuming standard column layout)
# Columns: [unit_id, cycle, op_setting_1, op_setting_2, op_setting_3, sensor_1, ..., sensor_21]
column_names = ['unit_id', 'cycle'] + [f'op_{i}' for i in range(1, 4)] + [f'sensor_{i}' for i in range(1, 22)]
train_data = pd.read_csv('./datasets/CMAPSS/train_FD001.txt', sep=r'\s+', header=None, names=column_names)

# 2. Extract a single engine unit (e.g., Unit 1) and two specific sensors: Sensor 2 and Sensor 4
engine_id = 1
engine_df = train_data[train_data['unit_id'] == engine_id]

t_obs = engine_df['cycle'].values
x1 = engine_df['sensor_2'].values  # Sensor 2: Total Temperature
x2 = engine_df['sensor_4'].values  # Sensor 4: HPC Pressure

# 3. Define failure thresholds (e.g., defined based on historical max/min tolerance limits)
L1 = np.percentile(x1, 95) # Soft failure boundary for sensor 1
L2 = np.percentile(x2, 95) # Soft failure boundary for sensor 2

class CMAPSSBivariateCopula:
    def __init__(self, alpha, L1, L2):
        self.alpha = alpha
        self.L1 = L1
        self.L2 = L2

    def gumbel_copula(self, u1, u2):
        u1 = np.clip(u1, 1e-10, 1 - 1e-10)
        u2 = np.clip(u2, 1e-10, 1 - 1e-10)
        term1 = (-np.log(u1)) ** self.alpha
        term2 = (-np.log(u2)) ** self.alpha
        return np.exp(-(term1 + term2) ** (1 / self.alpha))

    def evaluate_reliability_at_time(self, t, mu1, sigma1, mu2, sigma2):
        # Marginal CDFs evaluated at thresholds L1, L2 for a given time t state
        u1 = norm.cdf(self.L1, loc=mu1(t), scale=sigma1)
        u2 = norm.cdf(self.L2, loc=mu2(t), scale=sigma2)
        return self.gumbel_copula(u1, u2)

# Simple empirical drift approximations for the selected unit's sensors
mu1_func = lambda t: np.polyval(np.polyfit(t_obs, x1, 1), t)
mu2_func = lambda t: np.polyval(np.polyfit(t_obs, x2, 1), t)
sigma1_val = np.std(x1 - mu1_func(t_obs))
sigma2_val = np.std(x2 - mu2_func(t_obs))

# Initialize and test
model = CMAPSSBivariateCopula(alpha=2.0, L1=L1, L2=L2)
current_time_step = 50
reliability = model.evaluate_reliability_at_time(current_time_step, mu1_func, sigma1_val, mu2_func, sigma2_val)

print(f"Engine {engine_id} Joint Reliability at cycle {current_time_step}: {reliability:.4f}")

Engine 1 Joint Reliability at cycle 50: 0.9996


In [7]:
import numpy as np
from scipy.stats import norm
from scipy.integrate import quad

class BivariateCopulaDegradation:
    def __init__(self, alpha, L1, L2):
        """
        Parameters:
        alpha: Gumbel copula dependence parameter (alpha >= 1)
        L1, L2: Soft failure thresholds for component 1 and 2
        """
        self.alpha = alpha
        self.L1 = L1
        self.L2 = L2

    def _marginal_reliability(self, t, mu_drift, sigma_noise, threshold):
        """
        Computes P(X_j(t) < L_j) assuming X_j(t) ~ N(mu(t), sigma(t)^2).
        For this simulation, we assume a linear drift mu(t) = beta * t 
        and constant diffusion/noise sigma(t) = sigma.
        """
        # Replace with piece-wise transition logic if evaluating non-linear phases
        mu_t = mu_drift * t
        sigma_t = sigma_noise * np.sqrt(t) if t > 0 else 1e-5 
        
        # P(X < L) = CDF(L) given N(mu_t, sigma_t)
        return norm.cdf(threshold, loc=mu_t, scale=sigma_t)

    def gumbel_copula(self, u1, u2):
        """
        Evaluates the Gumbel copula joint CDF.
        Clips marginals to avoid log(0) or log(1) edge cases during numerical integration.
        """
        u1 = np.clip(u1, 1e-10, 1 - 1e-10)
        u2 = np.clip(u2, 1e-10, 1 - 1e-10)
        
        term1 = (-np.log(u1)) ** self.alpha
        term2 = (-np.log(u2)) ** self.alpha
        return np.exp(-(term1 + term2) ** (1 / self.alpha))

    def joint_reliability(self, t, params1, params2):
        """
        Computes R(t) = C(F_1(L1; t), F_2(L2; t); alpha)
        """
        u1 = self._marginal_reliability(t, params1['drift'], params1['noise'], self.L1)
        u2 = self._marginal_reliability(t, params2['drift'], params2['noise'], self.L2)
        return self.gumbel_copula(u1, u2)

    def expected_rul(self, t_star, params1, params2):
        """
        Computes E[RUL | T > t*] via numerical integration of the conditional survival function.
        """
        # Baseline reliability at current time t*
        r_t_star = self.joint_reliability(t_star, params1, params2)
        
        if r_t_star < 1e-5:
            return 0.0 # System has effectively already failed

        def conditional_survival_integrand(tau):
            return self.joint_reliability(tau, params1, params2) / r_t_star

        # Integrate from current time to a practical infinite horizon
        # quad handles np.inf, but upper limit of 1000 is safer for heavily tailed numerical approximations
        rul, error_estimate = quad(conditional_survival_integrand, t_star, np.inf, limit=200)
        
        return rul

# ==========================================
# Execution Example
# ==========================================

# 1. Define estimated parameters (e.g., extracted from IFM/MLE step)
comp1_params = {'drift': 0.15, 'noise': 0.05} # Component 1 degrades slower
comp2_params = {'drift': 0.25, 'noise': 0.08} # Component 2 degrades faster
dependence_alpha = 1.8 # alpha > 1 indicates positive upper-tail dependence
failure_thresholds = (5.0, 5.0)

# 2. Initialize the model
model = BivariateCopulaDegradation(alpha=dependence_alpha, 
                                   L1=failure_thresholds[0], 
                                   L2=failure_thresholds[1])

# 3. Evaluate at a specific observation time t*
t_star = 12.0

# 4. Compute metrics
current_reliability = model.joint_reliability(t_star, comp1_params, comp2_params)
predicted_rul = model.expected_rul(t_star, comp1_params, comp2_params)

print(f"System Reliability at t={t_star}: {current_reliability:.4f}")
print(f"Expected RUL from t={t_star}: {predicted_rul:.4f} time units")

System Reliability at t=12.0: 1.0000
Expected RUL from t=12.0: 8.0512 time units


In [11]:
# Load training data
train_data = pd.read_csv('./datasets/CMAPSS/train_FD001.txt', sep=r'\s+', header=None, names=column_names)

# Find the final cycle for each training engine
max_cycles = train_data.groupby('unit_id')['cycle'].max()

# Extract sensor values at the exact moment of failure for all training engines
failure_sensor_2 = []
failure_sensor_4 = []

for engine_id, last_cycle in max_cycles.items():
    engine_row = train_data[(train_data['unit_id'] == engine_id) & (train_data['cycle'] == last_cycle)]
    failure_sensor_2.append(engine_row['sensor_2'].values[0])
    failure_sensor_4.append(engine_row['sensor_4'].values[0])

# Set data-driven thresholds based on end-of-life behavior
L1_threshold = np.mean(failure_sensor_2) # Average temperature at failure
L2_threshold = np.mean(failure_sensor_4) # Average pressure at failure

print(f"Data-driven Threshold L1 (Sensor 2): {L1_threshold:.2f}")
print(f"Data-driven Threshold L2 (Sensor 4): {L2_threshold:.2f}")

Data-driven Threshold L1 (Sensor 2): 643.72
Data-driven Threshold L2 (Sensor 4): 1430.87


In [13]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.integrate import quad

# 1. Load C-MAPSS Test Data and Ground Truth RUL
column_names = ['unit_id', 'cycle'] + [f'op_{i}' for i in range(1, 4)] + [f'sensor_{i}' for i in range(1, 22)]
test_data = pd.read_csv('./datasets/CMAPSS/test_FD001.txt', sep=r'\s+', header=None, names=column_names)
true_rul = pd.read_csv('./datasets/CMAPSS/RUL_FD001.txt', sep=r'\s+', header=None).values.flatten()

class CMAPSSModelTester:
    def __init__(self, alpha, L1, L2):
        self.alpha = alpha
        self.L1 = L1
        self.L2 = L2

    def gumbel_copula(self, u1, u2):
        u1 = np.clip(u1, 1e-10, 1 - 1e-10)
        u2 = np.clip(u2, 1e-10, 1 - 1e-10)
        term1 = (-np.log(u1)) ** self.alpha
        term2 = (-np.log(u2)) ** self.alpha
        return np.exp(-(term1 + term2) ** (1 / self.alpha))

    def joint_reliability(self, t, mu1_func, sigma1, mu2_func, sigma2):
        u1 = norm.cdf(self.L1, loc=mu1_func(t), scale=sigma1)
        u2 = norm.cdf(self.L2, loc=mu2_func(t), scale=sigma2)
        return self.gumbel_copula(u1, u2)

    def predict_rul(self, t_star, mu1_func, sigma1, mu2_func, sigma2):
        r_t_star = self.joint_reliability(t_star, mu1_func, sigma1, mu2_func, sigma2)
        if r_t_star < 1e-5:
            return 0.0
        
        def conditional_survival_integrand(tau):
            return self.joint_reliability(tau, mu1_func, sigma1, mu2_func, sigma2) / r_t_star

        # Integrate up to a safe future horizon (e.g., current time + 300 cycles)
        horizon = t_star + 300
        rul, _ = quad(conditional_survival_integrand, t_star, horizon, limit=100)
        return max(0.0, rul)

# 2. Define operational failure thresholds (derived from max/min bounds of healthy/failed states)
# For FD001, Sensor 2 and Sensor 4 typical failure bounds can be approximated or set via training statistics
# L1_threshold = 645.0  # Threshold for Sensor 2 (Temperature)
# L2_threshold = 1390.0 # Threshold for Sensor 4 (Pressure)

tester = CMAPSSModelTester(alpha=1.8, L1=L1_threshold, L2=L2_threshold)

predictions = []
actuals = []

# 3. Loop through all 100 test engines in FD001
unique_engines = test_data['unit_id'].unique()

for idx, engine_id in enumerate(unique_engines):
    engine_df = test_data[test_data['unit_id'] == engine_id]
    
    t_obs = engine_df['cycle'].values
    x1 = engine_df['sensor_2'].values
    x2 = engine_df['sensor_4'].values
    
    t_star = t_obs[-1] # Truncated cycle point for this test engine
    
    # Fit linear drift trend up to the truncation point
    p1 = np.polyfit(t_obs, x1, 1)
    p2 = np.polyfit(t_obs, x2, 1)
    
    mu1_func = lambda t: p1[0] * t + p1[1]
    mu2_func = lambda t: p2[0] * t + p2[1]
    
    sigma1 = np.std(x1 - np.polyval(p1, t_obs)) + 1e-5
    sigma2 = np.std(x2 - np.polyval(p2, t_obs)) + 1e-5
    
    # Predict Remaining Useful Life
    pred_rul = tester.predict_rul(t_star, mu1_func, sigma1, mu2_func, sigma2)
    
    predictions.append(pred_rul)
    actuals.append(true_rul[idx])

# 4. Compute Evaluation Metrics (RMSE)
predictions = np.array(predictions)
actuals = np.array(actuals)

rmse = np.sqrt(np.mean((predictions - actuals) ** 2))
print(f"Evaluation Complete across {len(unique_engines)} test engines.")
print(f"Root Mean Square Error (RMSE): {rmse:.4f}")

Evaluation Complete across 100 test engines.
Root Mean Square Error (RMSE): 177.2393
